In [1]:
import scanpy as sc
import scvelo as scv
import pickle as pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from regvelo import REGVELOVI
import regvelo as rgv
import torch

## Gastrulation erythroid

In [2]:
adata = sc.read_h5ad('../Data/gastrulation_erythroid.h5ad')
TFT_table = pd.read_csv(f"../network/TFT_mouse.csv", index_col=0, header=0)

scv.pp.filter_genes(adata, min_shared_counts=20)
scv.pp.normalize_per_cell(adata)
scv.pp.filter_genes_dispersion(adata, n_top_genes=3000)
sc.pp.log1p(adata)

sc.pp.neighbors(adata, n_pcs=30, n_neighbors=30)
scv.pp.moments(adata, n_pcs=None, n_neighbors=None)

all_genes = adata.var_names
print(len(all_genes))
filtered_tft = TFT_table[
    TFT_table['from'].isin(all_genes) & 
    TFT_table['to'].isin(all_genes)
].copy()
print(filtered_tft.shape)

W_df = pd.DataFrame(0, index=all_genes, columns=all_genes)
W_df.values[W_df.index.get_indexer(filtered_tft['from']), W_df.columns.get_indexer(filtered_tft['to'])] = 1
adata.uns["skeleton"] = W_df

velocity_genes = rgv.pp.preprocess_data(adata.copy()).var_names.tolist()
TF = adata.var_names[adata.uns["skeleton"].sum(1) != 0]
var_mask = np.union1d(TF, velocity_genes)
adata = adata[:, var_mask].copy()
adata.uns["skeleton"] = adata.uns["skeleton"].loc[adata.var_names.tolist(), adata.var_names.tolist()]

adata = rgv.pp.filter_genes(adata)
adata = rgv.pp.preprocess_data(adata, filter_on_r2=False)
adata.var["velocity_genes"] = adata.var_names.isin(velocity_genes)
adata.var["TF"] = adata.var_names.isin(TF)

W = adata.uns["skeleton"].copy()
W = torch.tensor(np.array(W)).int()
REGVELOVI.setup_anndata(adata, spliced_layer="Ms", unspliced_layer="Mu")
vae = REGVELOVI(adata, W=W.T, regulators=TF)
vae.train()
vae.get_latent_representation()
vae.get_velocity()
vae.get_latent_time()
vae.history
rgv.tl.set_output(adata, vae, n_samples=30, batch_size=adata.n_obs)

scv.tl.velocity_graph(adata, vkey='velocity', n_jobs=-1)
scv.tl.velocity_embedding(adata, basis='umap', vkey='velocity')

fix, ax = plt.subplots(1, 1, figsize = (8, 6))
scv.pl.velocity_embedding_stream(adata, basis='umap', save = False, vkey='velocity', color='celltype', title='RegVelo', fontsize=20, arrow_size=2.0, arrow_color='#606060', show=False, ax=ax, legend_fontsize=15)
plt.savefig('../Gastrulation_Erythroid_Results/RegVelo_UMAP_stream.png', dpi=80)
adata.write_h5ad('../Gastrulation_Erythroid_Results/RegVelo.h5ad')

Filtered out 47456 genes that are detected 20 counts (shared).
Normalized count data: X, spliced, unspliced.
Extracted 3000 highly variable genes.
computing moments based on connectivities
    finished (0:00:01) --> added 
    'Ms' and 'Mu', moments of un/spliced abundances (adata.layers)
3000
(261778, 4)
computing velocities
    finished (0:00:00) --> added 
    'velocity', velocity vectors for each individual cell (adata.layers)
Number of genes: 953


/data/conda_env/lyw/3.10/lib/python3.10/site-packages/scvi/dataloaders/_data_splitting.py:210: UserWarning: Last batch will have a small size of 2samples. Consider changing settings.batch_size or batch_size in model.traincurrently 128 to avoid errors during model training.
  self.n_train, self.n_val = validate_data_split(
Trainer will use only 1 of 8 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=8)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA H100 80GB HBM3') that has Tensor Cores. To proper

Training:   0%|          | 0/1500 [00:00<?, ?it/s]

Monitored metric elbo_validation did not improve in the last 45 records. Best score: -2184.348. Signaling Trainer to stop.
computing velocity graph (using 192/192 cores)


  0%|          | 0/9815 [00:00<?, ?cells/s]

    finished (0:00:09) --> added 
    'velocity_graph', sparse matrix with cosine correlations (adata.uns)
computing velocity embedding
    finished (0:00:00) --> added
    'velocity_umap', embedded velocity vectors (adata.obsm)


## Dentate Gyrus neurogenesis

In [5]:
adata = sc.read_h5ad('../Data/DentateGyrus_10X43_1.h5ad')
TFT_table = pd.read_csv(f"../network/TFT_mouse.csv", index_col=0, header=0)

scv.pp.filter_genes(adata, min_shared_counts=20)
scv.pp.normalize_per_cell(adata)
scv.pp.filter_genes_dispersion(adata, n_top_genes=3000)
sc.pp.log1p(adata)

sc.pp.neighbors(adata, n_pcs=30, n_neighbors=30)
scv.pp.moments(adata, n_pcs=None, n_neighbors=None)

all_genes = adata.var_names

filtered_tft = TFT_table[
    TFT_table['from'].isin(all_genes) & 
    TFT_table['to'].isin(all_genes)
].copy()

W_df = pd.DataFrame(0, index=all_genes, columns=all_genes)
W_df.values[W_df.index.get_indexer(filtered_tft['from']), W_df.columns.get_indexer(filtered_tft['to'])] = 1
adata.uns["skeleton"] = W_df

velocity_genes = rgv.pp.preprocess_data(adata.copy()).var_names.tolist()
TF = adata.var_names[adata.uns["skeleton"].sum(1) != 0]
var_mask = np.union1d(TF, velocity_genes)
adata = adata[:, var_mask].copy()
adata.uns["skeleton"] = adata.uns["skeleton"].loc[adata.var_names.tolist(), adata.var_names.tolist()]

adata = rgv.pp.filter_genes(adata)
adata = rgv.pp.preprocess_data(adata, filter_on_r2=False)
adata.var["velocity_genes"] = adata.var_names.isin(velocity_genes)
adata.var["TF"] = adata.var_names.isin(TF)

W = adata.uns["skeleton"].copy()
W = torch.tensor(np.array(W)).int()
REGVELOVI.setup_anndata(adata, spliced_layer="Ms", unspliced_layer="Mu")
vae = REGVELOVI(adata, W=W.T, regulators=TF)
vae.train()
vae.get_latent_representation()
vae.get_velocity()
vae.get_latent_time()
vae.history
rgv.tl.set_output(adata, vae, n_samples=30, batch_size=adata.n_obs)

scv.tl.velocity_graph(adata, vkey='velocity', n_jobs=-1)
scv.tl.velocity_embedding(adata, basis='umap', vkey='velocity')

fix, ax = plt.subplots(1, 1, figsize = (8, 6))
scv.pl.velocity_embedding_stream(adata, basis='umap', save = False, vkey='velocity', color='clusters', title='RegVelo', fontsize=20, arrow_size=2.0, arrow_color='#A0A0A0', show=False, ax=ax, legend_fontsize=15)
plt.savefig('../Dentate_Gyrus_Results/RegVelo_UMAP_stream.png', dpi=80)
adata.write_h5ad('../Dentate_Gyrus_Results/RegVelo.h5ad')

Filtered out 10340 genes that are detected 20 counts (shared).
Normalized count data: X, spliced, unspliced.
Extracted 3000 highly variable genes.


/data/conda_env/lyw/3.10/lib/python3.10/site-packages/scanpy/neighbors/__init__.py:577: UserWarning: You’re trying to run this on 3000 dimensions of `.X`, if you really want this, set `use_rep='X'`.
         Falling back to preprocessing with `sc.pp.pca` and default params.
  x = _choose_representation(self._adata, use_rep=use_rep, n_pcs=n_pcs)


computing moments based on connectivities
    finished (0:00:00) --> added 
    'Ms' and 'Mu', moments of un/spliced abundances (adata.layers)
computing velocities
    finished (0:00:00) --> added 
    'velocity', velocity vectors for each individual cell (adata.layers)


Trainer will use only 1 of 8 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=8)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Number of genes: 1066


...

Training:   0%|          | 0/1500 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=1500` reached.


computing velocity graph (using 192/192 cores)


  0%|          | 0/2930 [00:00<?, ?cells/s]

    finished (0:00:08) --> added 
    'velocity_graph', sparse matrix with cosine correlations (adata.uns)
computing velocity embedding
    finished (0:00:00) --> added
    'velocity_umap', embedded velocity vectors (adata.obsm)


## Simulated data

In [4]:
adata = sc.read_h5ad('../Data/simulated_T_cell.h5ad')
adata.obsm['X_umap'] = pickle.load(open('../Data/UMAP.pkl','rb'))

sc.pp.neighbors(adata, n_pcs=30, n_neighbors=30)
scv.pp.moments(adata, n_neighbors=30)

W_df = pd.DataFrame(0, index=adata.var_names, columns=adata.var_names)
W_df.loc['KRAS', 'MAPK1'] = 1
W_df.loc['PTPN11', 'MAPK1'] = 1
W_df.loc['PTPN11', 'KRAS'] = 1
adata.uns["skeleton"] = W_df

velocity_genes = adata.var_names.tolist()
TF = adata.var_names[adata.uns["skeleton"].sum(1) != 0]
adata.var["velocity_genes"] = adata.var_names.isin(velocity_genes)
adata.var["TF"] = adata.var_names.isin(TF)

W = adata.uns["skeleton"].copy()
W = torch.tensor(np.array(W)).int()
REGVELOVI.setup_anndata(adata, spliced_layer="Ms", unspliced_layer="Mu")
vae = REGVELOVI(adata, W=W.T, regulators=TF)
vae.train()
vae.get_latent_representation()
vae.get_velocity()
vae.get_latent_time()
vae.history
rgv.tl.set_output(adata, vae, n_samples=30, batch_size=adata.n_obs)

scv.tl.velocity_graph(adata, vkey='velocity', n_jobs=-1)
scv.tl.velocity_embedding(adata, basis='umap', vkey='velocity')

fix, ax = plt.subplots(1, 1, figsize = (8, 6))
scv.pl.velocity_embedding_stream(adata, basis='umap', save = False, vkey='velocity', color='time', title='RegVelo', fontsize=20, show = False, ax = ax, legend_fontsize = 15)
plt.savefig('../Simulation_Results/RegVelo_UMAP_stream.png', dpi=80)
adata.write_h5ad('../Simulation_Results/RegVelo.h5ad')

Trainer will use only 1 of 8 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=8)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


computing moments based on connectivities
    finished (0:00:00) --> added 
    'Ms' and 'Mu', moments of un/spliced abundances (adata.layers)


/data/conda_env/lyw/3.10/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=191` in the `DataLoader` to improve performance.
/data/conda_env/lyw/3.10/lib/python3.10/site-packages/lightning/pytorch/loops/fit_loop.py:317: The number of training batches (1) is smaller than the logging interval Trainer(log_every_n_steps=10). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.
/data/conda_env/lyw/3.10/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=191` in the `DataLoader` to improve performance.
/data/conda_env/lyw/3.10/lib/python3.10/site-packages/lightning/pytorch/loops/fit_

Training:   0%|          | 0/1500 [00:00<?, ?it/s]

Monitored metric elbo_validation did not improve in the last 45 records. Best score: 6.083. Signaling Trainer to stop.
computing velocity graph (using 192/192 cores)


  0%|          | 0/4000 [00:00<?, ?cells/s]

    finished (0:00:01) --> added 
    'velocity_graph', sparse matrix with cosine correlations (adata.uns)
computing velocity embedding
    finished (0:00:00) --> added
    'velocity_umap', embedded velocity vectors (adata.obsm)
